In [5]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors

In [7]:
movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data/ratings.csv")

In [8]:
movies.shape

(9742, 3)

In [9]:
ratings.shape

(100836, 4)

In [10]:
movie_ratings = ratings.merge(
    movies,
    on='movieId'
)

In [11]:
movie_ratings.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [12]:
movie_stats = movie_ratings.groupby(
    'title'
)['rating'].agg(['mean','count'])

In [13]:
movie_stats.head()

,mean,count
title,,
'71 (2014),4.0,1
'Hellboy': The Seeds of Creation (2004),4.0,1
'Round Midnight (1986),3.5,2
'Salem's Lot (2004),5.0,1
'Til There Was You (1997),4.0,2


In [14]:
popular_movies = movie_stats[
    movie_stats['count'] >= 100
].sort_values(
    'mean',
    ascending=False
)

In [15]:
popular_movies.head(10)

,mean,count
title,,
"Shawshank Redemption, The (1994)",4.429022,317
"Godfather, The (1972)",4.289062,192
Fight Club (1999),4.272936,218
"Godfather: Part II, The (1974)",4.259690,129
"Departed, The (2006)",4.252336,107
Goodfellas (1990),4.250000,126
Casablanca (1942),4.240000,100
"Dark Knight, The (2008)",4.238255,149
"Usual Suspects, The (1995)",4.237745,204


In [16]:
user_movie_matrix = movie_ratings.pivot_table(
    index='title',
    columns='userId',
    values='rating'
)

In [17]:
user_movie_matrix.shape

(9719, 610)

In [18]:
user_movie_matrix_filled = user_movie_matrix.fillna(0)

In [19]:
user_movie_matrix_filled.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
'Hellboy': The Seeds of Creation (2004),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Round Midnight (1986),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Salem's Lot (2004),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Til There Was You (1997),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
knn = NearestNeighbors(
    metric='cosine',
    algorithm='brute'
)

knn.fit(user_movie_matrix_filled)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [21]:
def recommend_collaborative(movie_name):

    movie_index = user_movie_matrix_filled.index.get_loc(movie_name)

    distances, indices = knn.kneighbors(
        user_movie_matrix_filled.iloc[movie_index].values.reshape(1, -1),
        n_neighbors=6
    )

    recommendations = []

    for i in range(1, len(indices[0])):
        recommendations.append(
            user_movie_matrix_filled.index[indices[0][i]]
        )

    return recommendations

In [22]:
recommend_collaborative("Toy Story (1995)")

['Toy Story 2 (1999)',
 'Jurassic Park (1993)',
 'Independence Day (a.k.a. ID4) (1996)',
 'Star Wars: Episode IV - A New Hope (1977)',
 'Forrest Gump (1994)']

In [23]:
def hybrid_recommend(movie_name):

    collaborative = recommend_collaborative(movie_name)

    popular = list(
        popular_movies.index[:5]
    )

    recommendations = []

    for movie in collaborative:
        recommendations.append(movie)

    for movie in popular:
        if movie not in recommendations:
            recommendations.append(movie)

    return recommendations[:10]

In [24]:
hybrid_recommend("Toy Story (1995)")

['Toy Story 2 (1999)',
 'Jurassic Park (1993)',
 'Independence Day (a.k.a. ID4) (1996)',
 'Star Wars: Episode IV - A New Hope (1977)',
 'Forrest Gump (1994)',
 'Shawshank Redemption, The (1994)',
 'Godfather, The (1972)',
 'Fight Club (1999)',
 'Godfather: Part II, The (1974)',
 'Departed, The (2006)']

In [33]:
genres = movies['genres'].str.get_dummies(sep='|')

movie_features = genres

In [38]:
def recommend_content(movie_name):

    movie_index = movies[
        movies['title'].str.contains(
            movie_name,
            case=False
        )
    ].index[0]

    distances, indices = content_knn.kneighbors(
        movie_features.iloc[movie_index].values.reshape(1, -1)
    )

    recommendations = []

    for idx in indices[0][1:]:
        recommendations.append(
            movies.iloc[idx]['title']
        )

    return recommendations

In [39]:
recommend_content("Toy Story")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(


['Monsters, Inc. (2001)',
 'Moana (2016)',
 'Asterix and the Vikings (Astérix et les Vikings) (2006)',
 'Turbo (2013)',
 'The Good Dinosaur (2015)']

In [36]:
from sklearn.neighbors import NearestNeighbors

content_knn = NearestNeighbors(
    n_neighbors=6,
    metric='cosine'
)

content_knn.fit(movie_features)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",6
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [40]:
recommend_content("Toy Story")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(


['Monsters, Inc. (2001)',
 'Moana (2016)',
 'Asterix and the Vikings (Astérix et les Vikings) (2006)',
 'Turbo (2013)',
 'The Good Dinosaur (2015)']

In [41]:
def hybrid_recommend(movie_name):

    content_recs = recommend_content(movie_name)

    collaborative_recs = recommend_collaborative(
        movie_name + " (1995)"
        if movie_name == "Toy Story"
        else movie_name
    )

    recommendations = []

    for movie in content_recs:
        if movie not in recommendations:
            recommendations.append(movie)

    for movie in collaborative_recs:
        if movie not in recommendations:
            recommendations.append(movie)

    return recommendations[:10]

In [42]:
hybrid_recommend("Toy Story")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(


['Monsters, Inc. (2001)',
 'Moana (2016)',
 'Asterix and the Vikings (Astérix et les Vikings) (2006)',
 'Turbo (2013)',
 'The Good Dinosaur (2015)',
 'Toy Story 2 (1999)',
 'Jurassic Park (1993)',
 'Independence Day (a.k.a. ID4) (1996)',
 'Star Wars: Episode IV - A New Hope (1977)',
 'Forrest Gump (1994)']

In [43]:
def hybrid_recommend(movie_name):

    content_recs = recommend_content(movie_name)

    collaborative_recs = recommend_collaborative(
        movies[
            movies['title'].str.contains(
                movie_name,
                case=False
            )
        ]['title'].iloc[0]
    )

    final_recommendations = []

    for movie in content_recs:
        if movie not in final_recommendations:
            final_recommendations.append(movie)

    for movie in collaborative_recs:
        if movie not in final_recommendations:
            final_recommendations.append(movie)

    return final_recommendations[:10]

In [44]:
hybrid_recommend("Toy Story")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(


['Monsters, Inc. (2001)',
 'Moana (2016)',
 'Asterix and the Vikings (Astérix et les Vikings) (2006)',
 'Turbo (2013)',
 'The Good Dinosaur (2015)',
 'Toy Story 2 (1999)',
 'Jurassic Park (1993)',
 'Independence Day (a.k.a. ID4) (1996)',
 'Star Wars: Episode IV - A New Hope (1977)',
 'Forrest Gump (1994)']

In [45]:
hybrid_recommend("Jumanji")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(


['NeverEnding Story, The (1984)',
 'Chronicles of Narnia: The Lion, the Witch and the Wardrobe, The (2005)',
 'Percy Jackson: Sea of Monsters (2013)',
 'NeverEnding Story II: The Next Chapter, The (1990)',
 'Chronicles of Narnia: Prince Caspian, The (2008)',
 'Lion King, The (1994)',
 'Mrs. Doubtfire (1993)',
 'Mask, The (1994)',
 'Jurassic Park (1993)',
 'Home Alone (1990)']

In [46]:
hybrid_recommend("Heat")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(


['Poker Night (2014)',
 'U.S. Marshals (1998)',
 'Hitman: Agent 47 (2015)',
 'John Wick: Chapter Two (2017)',
 'Dead Pool, The (1988)',
 'Rock, The (1996)',
 'Twelve Monkeys (a.k.a. 12 Monkeys) (1995)',
 'Léon: The Professional (a.k.a. The Professional) (Léon) (1994)',
 'Casino (1995)',
 'Fargo (1996)']